# Latency EVT (GEV) — Quick Notebook
This notebook shows how to:
1) Audit a Parquet directory fast (metadata only)
2) Compute throughput (msgs/s, MB/s)
3) Build block maxima from latency and fit a GEV model
4) Summarize/plot results

**Assumptions**: Your Parquet schema includes columns:
`producer_timestamp`, `consumer_receive_timestamp`,
`application_latency_seconds`, `end_to_end_latency_seconds`, `size_bytes`, ...


In [2]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path

# Import tools from your cleaned module
from latency_evt_tools_clean import (
    audit_from_metadata,
    throughput_from_parquet_dir_dataset,
    collect_block_maxima_from_metadata,
    fit_gev, gev_params_dict, gev_quantiles,
)

# Optional: matplotlib for a quick chart
import matplotlib.pyplot as plt


## 1) Point to your Parquet root

In [3]:
#root = "../results/20250918_165513/consumer/consumer-sts-0_consumer-result/"
root = Path("../results/20250921_210953/consumer/consumer-sts-0_consumer-result/")  


assert root.exists(), f"Not found: {root}"

## 2) Fast audit (no column reads)

In [4]:
audit = audit_from_metadata(root, time_col="consumer_receive_timestamp")
audit

{'files_scanned': 3867,
 'rows_total': 386700,
 'tmin': Timestamp('2025-09-22 02:09:50.133450496+0000', tz='UTC'),
 'tmax': Timestamp('2025-09-22 03:08:03.753273600+0000', tz='UTC'),
 'span_sec': 3493.619823,
 'note': 'metadata-only (no data scan)'}

## 3) Fast throughput using Arrow Dataset

In [5]:
ts = throughput_from_parquet_dir_dataset(
    root,
    time_col="consumer_receive_timestamp",
    size_col="size_bytes",
    time_unit="s",
)
ts.describe()

ArrowInvalid: Float value 1.75851e+09 was truncated converting to int64

In [6]:
# Plot (optional)
ts['MB_per_sec'].plot(figsize=(10,3))
plt.title('Throughput (MB/s)'); plt.xlabel('time'); plt.ylabel('MB/s'); plt.show()

NameError: name 'ts' is not defined

## 4) Block maxima from latency and GEV fit

In [7]:
# Collect per-file block maxima from footer stats (very fast)
lat_col = "end_to_end_latency_seconds"  # or "application_latency_seconds"
bm = collect_block_maxima_from_metadata(root, latency_col=lat_col)
len(bm), float(np.nanmean(bm)), float(np.nanmax(bm))

(3867, 0.6114483347361743, 51.8912014585674)

In [8]:
# Fit GEV and compute tail quantiles
c, loc, scale = fit_gev(bm)
params = gev_params_dict(c, loc, scale)
q = gev_quantiles([0.95, 0.99, 0.999], c, loc, scale)
params, q

({'shape_c': -0.1462032907870014,
  'xi': 0.1462032907870014,
  'loc': 0.5220300321376641,
  'scale': 0.012174785798549649},
 {0.95: 0.5673145819458393,
  0.99: 0.6019088701014834,
  0.999: 0.6673602311027756})

In [9]:
# (Optional) Return level for "T blocks"; e.g., expected max once every 1,000 blocks
from latency_evt_tools_clean import return_level
return_level(1000, c, loc, scale)

0.6673602311027756

## 5) Simple text summary

In [10]:
summary = {
    "files_scanned": audit.get("files_scanned"),
    "rows_total": audit.get("rows_total"),
    "tmin": str(audit.get("tmin")),
    "tmax": str(audit.get("tmax")),
    "span_sec": audit.get("span_sec"),
    "throughput_MBps_mean": float(ts["MB_per_sec"].mean()) if not ts.empty else None,
    "throughput_msgsps_mean": float(ts["msgs_per_sec"].mean()) if not ts.empty else None,
    "GEV_params": params,
    "GEV_q": q,
}
summary

NameError: name 'ts' is not defined